# Quantized LLM Evaluation on Colab
## 4-Model Core Set × 5 Quantization Precisions

**Available Precisions:**
- `int8` — 8-bit integer (universal support)
- `int4` — 4-bit integer (TensorRT, ONNX, broader edge device support)
- `nf4` — 4-bit Normal Float (NVIDIA GPUs only)
- `fp8` — 8-bit floating-point (modern GPUs + emerging edge support)

**GPU / TPU Compatibility:**
- ✅ NF4 supported: H100, G4, A100, L4, T4 GPUs
- ❌ NF4 NOT supported: v6e-1 TPU, v5e-1 TPU
- ⚠️ Edge devices (RPi5, Jetson Xavier/Orin, Qualcomm): Limited/no standard NF4 support

**Recommended Setup:** L4 GPU on Colab


In [ ]:
import os
import platform
import subprocess
import sys

import torch

print("=" * 70)
print("ENVIRONMENT & GPU DIAGNOSTICS")
print("=" * 70)
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print()

print("CUDA / GPU:")
print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"  torch.cuda.device_count(): {torch.cuda.device_count()}")
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    print(f"  Current device index: {idx}")
    print(f"  GPU name: {torch.cuda.get_device_name(idx)}")
    print(f"  Compute capability: {props.major}.{props.minor}")
    print(f"  Total VRAM (GiB): {props.total_memory / (1024**3):.2f}")
else:
    print("  No CUDA GPU detected by PyTorch.")
print()

print("TPU / Optional accelerators:")
print(f"  TPU_NAME env: {os.environ.get('TPU_NAME', '(not set)')}")
print(f"  COLAB_TPU_ADDR env: {os.environ.get('COLAB_TPU_ADDR', '(not set)')}")
print()

print("nvidia-smi:")
try:
    out = subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT, text=True)
    print(out)
except Exception as e:
    print(f"  nvidia-smi unavailable: {e}")

print("=" * 70)


## Step 1: Clone Repository & Install Dependencies


In [ ]:
import os
import subprocess

# Clone the repository if not already present
repo_dir = "/content/intent-classifier"
if not os.path.isdir(repo_dir):
    print("Cloning intent-classifier repository…")
    subprocess.run(
        ["git", "clone", "https://github.com/kon172verma/intent-classifier.git", repo_dir],
        check=True,
    )
    print("✓ Repository cloned successfully.\n")
else:
    print(f"✓ Repository already present at {repo_dir}\n")

# Install Python dependencies
print("Installing Python dependencies…")
subprocess.run(
    [
        "pip", "install", "-q",
        "transformers",
        "torch",
        "bitsandbytes",
        "python-dotenv",
        "matplotlib",
        "numpy",
    ],
    check=True,
)
print("✓ Dependencies installed.\n")

# Change to the intent-classifier directory
os.chdir(repo_dir)
print(f"✓ Working directory: {os.getcwd()}\n")


## Step 2: Set Up Hugging Face Token
Authenticate with Hugging Face Hub to access gated models (if needed).


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Try to load from .env file
env_file = "/content/intent-classifier/.env"
if os.path.exists(env_file):
    load_dotenv(env_file)
    hf_token = os.getenv("HF_TOKEN")
    if hf_token:
        print("Loaded HF_TOKEN from .env file.")
        login(token=hf_token)
        print("✓ Authenticated with Hugging Face Hub.\n")
    else:
        print("HF_TOKEN not found in .env. Skipping HF authentication.\n")
else:
    print(f".env file not found at {env_file}. Skipping HF authentication.\n")
    print("Note: You can set HF_TOKEN in a Colab Secret or manually via login().\n")


## Step 3: Configuration & Model Definitions
Single source of truth for models (4-core set) and quantization precisions (6 variants).


In [ ]:
# ── 4-MODEL CORE SET ───────────────────────────────────────────────────────
SELECTED_MODELS = {
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen3-0.6b": "Qwen/Qwen3-0.6B",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "smollm3": "HuggingFaceTB/SmolLM3-3B",
}

MODEL_DISPLAY = {
    "qwen2.5-0.5b": "Qwen2.5-0.5B (~500M)",
    "qwen3-0.6b": "Qwen3-0.6B (~600M)",
    "qwen2.5-1.5b": "Qwen2.5-1.5B (~1.5B)",
    "smollm3": "SmolLM3-3B (~3.0B)",
}

# ── 5 QUANTIZATION PRECISIONS ──────────────────────────────────────────────
QUANTIZATION_PRECISIONS = {
    "int8": "8-bit integer (universal)",
    "int4": "4-bit integer (TensorRT, ONNX compatible)",
    "nf4": "4-bit Normal Float (NVIDIA GPUs only)",
    "fp8": "8-bit float (modern GPUs + edge)",
}

# ── DEFAULT PATHS ──────────────────────────────────────────────────────────
DEFAULT_DATA_PATH = "/content/intent-classifier/dataset_sample/sample.json"
QUANTIZED_EVAL_DIR = "/content/intent-classifier/quantized_evaluation"

print("=" * 70)
print("CONFIGURATION: 4 Models × 5 Quantization Precisions")
print("=" * 70)
print("\nModels:")
for key, model_display in MODEL_DISPLAY.items():
    print(f"  - {model_display}")

print("\nQuantization Precisions:")
for prec, desc in QUANTIZATION_PRECISIONS.items():
    print(f"  - {prec:<8} : {desc}")

print(f"\nData path: {DEFAULT_DATA_PATH}")
print(f"Output dir: {QUANTIZED_EVAL_DIR}")
print("=" * 70 + "\n")


## Step 4: NF4 Compatibility & Hardware Support Matrix


In [ ]:
print("=" * 80)
print("NF4 QUANTIZATION HARDWARE SUPPORT MATRIX")
print("=" * 80)
print()

# GPU support
print("DATA CENTER & CLOUD GPUs (NF4 Support via CUDA + BitsAndBytes):")
gpus_supported = ["H100 GPU", "G4 GPU", "A100 GPU", "L4 GPU", "T4 GPU"]
for gpu in gpus_supported:
    print(f"  ✅ {gpu:<20} Full support for NF4 quantization")
print()

# TPU support
print("TPUs (NF4 Support Status):")
tpus_unsupported = ["v6e-1 TPU", "v5e-1 TPU"]
for tpu in tpus_unsupported:
    print(f"  ❌ {tpu:<20} No native NF4 support (uses bfloat16/int8 instead)")
print()

# Edge devices
print("EDGE DEVICES (NF4 Support Status):")
print("  ⚠️  Raspberry Pi 5      NOT supported (no NVIDIA BitsAndBytes)")
print("  ⚠️  NVIDIA Jetson Xavier Limited (TensorRT support is partial)")
print("  ⚠️  NVIDIA Jetson Orin  Limited (TensorRT support is partial)")
print("  ❌ Qualcomm devices    NOT supported (QNN uses proprietary format)")
print()

print("=" * 80)
print()

print("RECOMMENDATION FOR COLAB:")
print("  This notebook runs on cloud GPUs (H100, A100, L4, T4).")
print("  NF4 quantization is fully supported on these platforms.")
print("  Edge device evaluation will require deployment-specific handling.")
print()
print("=" * 80)
print()


## Step 5: Run Quantization Evaluation (Smoke Test)
Quick test: run the smallest model (`qwen2.5-0.5b`) across 2 quantization precisions (int8, int4) on a subset of data (10 examples).


In [ ]:
import subprocess
import sys
from pathlib import Path
import time
import torch

# Configuration for smoke test
SMOKE_TEST_MODEL = "qwen2.5-0.5b"
SMOKE_TEST_QUANTS = ["int8", "int4"]
SMOKE_TEST_LIMIT = 10  # Evaluate on 10 examples only

eval_script = Path(QUANTIZED_EVAL_DIR) / "src" / "quant_eval.py"

# Helper function to get current GPU info
def get_gpu_info():
    """Return current GPU name and index if available."""
    if torch.cuda.is_available():
        device_idx = torch.cuda.current_device()
        device_name = torch.cuda.get_device_name(device_idx)
        return device_idx, device_name
    return None, "CPU"

print("=" * 90)
print("QUANTIZATION EVALUATION SMOKE TEST")
print("=" * 90)
print(f"Model to evaluate: {SELECTED_MODELS.get(SMOKE_TEST_MODEL, 'Unknown')}")
print(f"Model display name: {MODEL_DISPLAY.get(SMOKE_TEST_MODEL, 'Unknown')}")
print(f"Quantization precisions: {', '.join(SMOKE_TEST_QUANTS)}")
print(f"Number of examples: {SMOKE_TEST_LIMIT} (full: 5000+)")
print(f"Evaluation mode: zero_shot")
print("=" * 90)
print()

dev_idx, dev_name = get_gpu_info()
print(f"GPU Information:")
print(f"  Device index: {dev_idx if dev_idx is not None else 'N/A'}")
print(f"  Device name: {dev_name}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(dev_idx)
    print(f"  Compute capability: {props.major}.{props.minor}")
    print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
print()
print("=" * 90)
print()

for idx, quant in enumerate(SMOKE_TEST_QUANTS, 1):
    dev_idx, dev_name = get_gpu_info()
    
    print(f"[{idx}/{len(SMOKE_TEST_QUANTS)}] Starting evaluation...")
    print(f"  Model: {SELECTED_MODELS.get(SMOKE_TEST_MODEL, 'Unknown')}")
    print(f"  Precision: {quant.upper()}")
    print(f"  GPU: {dev_name} (device {dev_idx if dev_idx is not None else 'N/A'})")
    print(f"  Examples: {SMOKE_TEST_LIMIT}")
    print()
    
    start_time = time.time()
    cmd = [
        sys.executable, str(eval_script),
        "--model", SMOKE_TEST_MODEL,
        "--quant", quant,
        "--mode", "zero_shot",
        "--limit", str(SMOKE_TEST_LIMIT),
        "--device", "auto",
    ]
    
    try:
        result = subprocess.run(cmd, cwd=QUANTIZED_EVAL_DIR, capture_output=False)
        elapsed = time.time() - start_time
        
        if result.returncode == 0:
            print(f"\n✅ COMPLETED | Precision: {quant.upper()} | Time: {elapsed:.1f}s")
        else:
            print(f"\n❌ FAILED | Precision: {quant.upper()} | Exit code: {result.returncode} | Time: {elapsed:.1f}s")
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"\n❌ ERROR | Precision: {quant.upper()} | Exception: {e} | Time: {elapsed:.1f}s")
    
    print("-" * 90)
    print()

print("=" * 90)
print("SMOKE TEST SUMMARY")
print("=" * 90)
print(f"Model evaluated: {SELECTED_MODELS.get(SMOKE_TEST_MODEL, 'Unknown')}")
print(f"Precisions tested: {', '.join(SMOKE_TEST_QUANTS)}")
print(f"Results saved to:")
for quant in SMOKE_TEST_QUANTS:
    report_dir = f"reports_{quant}" if quant != "bf16" else "reports_zero_shot"
    print(f"  - {QUANTIZED_EVAL_DIR}/{report_dir}/")
print("=" * 90)


## Step 6: Generate Comparison Plots


In [ ]:
import subprocess
import sys
from pathlib import Path
import time
import torch

plot_script = Path(QUANTIZED_EVAL_DIR) / "src" / "plot_quant.py"

def get_gpu_info():
    """Return current GPU name and index if available."""
    if torch.cuda.is_available():
        device_idx = torch.cuda.current_device()
        device_name = torch.cuda.get_device_name(device_idx)
        return device_idx, device_name
    return None, "CPU"

dev_idx, dev_name = get_gpu_info()

print("=" * 90)
print("GENERATING QUANTIZATION COMPARISON PLOTS")
print("=" * 90)
print(f"GPU: {dev_name} (device {dev_idx if dev_idx is not None else 'N/A'})")
print(f"Plot script: {plot_script}")
print(f"Output directory: {Path(QUANTIZED_EVAL_DIR) / 'analysis'}")
print("=" * 90)
print()

start_time = time.time()

cmd = [
    sys.executable, str(plot_script),
    "--out-dir", str(Path(QUANTIZED_EVAL_DIR) / "analysis"),
]

print("Executing plot generation script...")
print()

result = subprocess.run(cmd, cwd=QUANTIZED_EVAL_DIR, capture_output=False)
elapsed = time.time() - start_time

print()
print("=" * 90)

if result.returncode == 0:
    print(f"✅ PLOT GENERATION COMPLETED | Time: {elapsed:.1f}s")
    print("=" * 90)
    print()
    
    analysis_dir = Path(QUANTIZED_EVAL_DIR) / "analysis"
    if analysis_dir.exists():
        pngs = sorted(analysis_dir.glob("*.png"))
        if pngs:
            print("Generated plots:")
            for i, img in enumerate(pngs, 1):
                file_size = img.stat().st_size / (1024 * 1024)  # Convert to MB
                print(f"  [{i}] {img.name} ({file_size:.2f} MB)")
        else:
            print("⚠️  No PNG files found in analysis directory")
    else:
        print("⚠️  Analysis directory does not exist")
else:
    print(f"❌ PLOT GENERATION FAILED | Exit code: {result.returncode} | Time: {elapsed:.1f}s")

print()
print("=" * 90)


## Step 7: Download Results (Optional)
If running on Colab, download evaluation reports and plots.


In [ ]:
import os
import shutil
from pathlib import Path

print("=" * 90)
print("PREPARING RESULTS FOR DOWNLOAD")
print("=" * 90)
print()

# Check if we're on Colab
try:
    from google.colab import files
    IS_COLAB = True
    print("✓ Detected Google Colab environment - Download available")
except ImportError:
    IS_COLAB = False
    print("ℹ Not running on Colab - Results remain in workspace\n")
    print("Results locations:")
    for quant in ["int8", "int4", "nf4", "fp8"]:
        report_dir = Path(QUANTIZED_EVAL_DIR) / f"reports_{quant}"
        if report_dir.exists():
            report_count = len(list(report_dir.glob("*.json")))
            print(f"  [{report_count} reports] {QUANTIZED_EVAL_DIR}/reports_{quant}/")
    
    analysis_dir = Path(QUANTIZED_EVAL_DIR) / "analysis"
    if analysis_dir.exists():
        plot_count = len(list(analysis_dir.glob("*.png")))
        print(f"  [{plot_count} plots]   {QUANTIZED_EVAL_DIR}/analysis/")
    print()

if IS_COLAB:
    print("Starting download preparation...\n")
    
    import shutil
    
    results_dir = Path("/tmp/quantized_eval_results")
    results_dir.mkdir(exist_ok=True)
    
    # Copy reports
    report_dirs = [
        "reports_int8",
        "reports_int4",
        "reports_nf4",
        "reports_fp8",
    ]
    
    copied_dirs = []
    for report_dir in report_dirs:
        src = Path(QUANTIZED_EVAL_DIR) / report_dir
        if src.exists() and list(src.glob("*.json")):
            dst = results_dir / report_dir
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            report_count = len(list(dst.glob("*.json")))
            print(f"  ✓ {report_dir}/ ({report_count} reports)")
            copied_dirs.append(report_dir)
    
    # Copy analysis (plots)
    analysis_src = Path(QUANTIZED_EVAL_DIR) / "analysis"
    if analysis_src.exists() and list(analysis_src.glob("*.png")):
        analysis_dst = results_dir / "analysis"
        if analysis_dst.exists():
            shutil.rmtree(analysis_dst)
        shutil.copytree(analysis_src, analysis_dst)
        plot_count = len(list(analysis_dst.glob("*.png")))
        print(f"  ✓ analysis/ ({plot_count} plots)")
        copied_dirs.append("analysis")
    
    if not copied_dirs:
        print("  ⚠️  No results found to download")
    else:
        print(f"\nCreating archive (quantized_eval_results.zip)...")
        
        # Archive and download
        archive = shutil.make_archive(
            "/tmp/quantized_eval_results",
            "zip",
            results_dir.parent,
            results_dir.name,
        )
        
        archive_size = Path(archive).stat().st_size / (1024 * 1024)
        print(f"  Archive size: {archive_size:.2f} MB")
        print()
        print("Downloading...")
        files.download(archive)
        print("\n✅ Download complete!")

print()
print("=" * 90)


## Summary & Next Steps

### ✅ What This Notebook Does
1. **Clones & sets up** the intent-classifier repository in Colab
2. **Installs dependencies** (transformers, torch, bitsandbytes, matplotlib)
3. **Authenticates** with Hugging Face Hub
4. **Runs smoke test** on the smallest model (qwen2.5-0.5b) with 2 precisions
5. **Generates plots** comparing quantization performance
6. **Downloads results** as a ZIP archive (Colab only)

### 🎯 Configuration
- **Models**: 4-core set (Qwen2.5-0.5B, Qwen3-0.6B, Qwen2.5-1.5B, SmolLM3-3B)
- **Quantizations**: 5 precisions (int8, int4, nf4, fp8)
- **Evaluation metrics**: Accuracy (%), latency (ms), throughput (tokens/s), peak memory (MB)

### 📊 Hardware Support
- **NVIDIA Cloud GPUs** (H100, A100, L4, T4): Full NF4 support ✅
- **TPUs**: Limited (falls back to bfloat16/int8)
- **Edge devices** (RPi5, Jetson, Qualcomm): Partial (int4/fp8 recommended)

### 🚀 To Run Full Evaluation
Replace smoke test parameters with:
- `--limit` 5000 (or higher)
- Expand `SMOKE_TEST_QUANTS` to include all 5 quantization types: `["int8", "int4", "nf4", "fp8"]`
- Run `run_quant_eval.py` for all 4 models × 4 precisions × 2 modes

### 📚 References
- **Repository**: https://github.com/kon172verma/intent-classifier
- **Evaluation Scripts**: `quantized_evaluation/src/`
  - `quant_eval.py` — Single model evaluation
  - `run_quant_eval.py` — Batch orchestration
  - `plot_quant.py` — Comparative plotting
- **Quantization Library**: [BitsAndBytes](https://github.com/TimDettmers/bitsandbytes)
- **Hardware Docs**: [NVIDIA Compute Capability Matrix](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#compute-capabilities)
